# Deterministic Verification

This notebook demonstrates the calculation of deterministic metrics for weather model evaluation.
Deterministic metrics assess the performance of a single model forecast (or an ensemble mean) against a ground truth (analysis or observation).

## Features

- **Overview and setup**: Describes the project helpers and configuration used to compare prediction outputs against target.
- **Maps**: Shows spatial side-by-side maps of target, prediction, and their difference (bias) to inspect geographic patterns.
- **Histograms**: Compares variable distributions between target and prediction using common binning for fair comparison.
- **Wasserstein Distance & KDE**: Computes Wasserstein distances and plots KDEs on standardized data to quantify distributional differences.
- **Energy Spectra**: Analyzes energy across spatial scales via FFT to evaluate model performance at different wavelengths.
- **Vertical Profiles**: Plots how errors vary with altitude across latitude bands for 3D variables.
- **Deterministic Metrics**: Calculates scalar metrics (MAE, MSE, RMSE, Pearson correlation, FSS) to summarize model accuracy.
- **Equitable Threat Score (ETS)**: Assesses threshold-based event skill while accounting for hits expected by chance.


## Relevant Code

The core logic resides in `src/swissclim_evaluations/metrics/deterministic.py`.

Key functions:
*   [`_calculate_all_metrics`](../src/swissclim_evaluations/metrics/deterministic.py#L136): Computes scalar metrics for a given dataset.
*   [`_calculate_per_level_metrics`](../src/swissclim_evaluations/metrics/deterministic.py#L450): Iterates over pressure levels for 3D variables.

## Configuration

Please specify the path to your configuration file below.

In [ ]:
# Define the path to the configuration file
config_path_str = "config/dev/full_verification_run.yaml"

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

from swissclim_evaluations.cli import (
    _load_yaml,
    _select_plot_datetime,
    _select_plot_ensemble,
    prepare_datasets,
    run_selected,
)
from swissclim_evaluations.helpers import display_outputs

##  Foundation Model Verification

This notebook verifies prediction outputs against target references using the SwissClim Evaluations library. The implementation now delegates all heavy lifting to the codebase modules (via `swissclim_evaluations.cli` and its plotting/metrics submodules) and preserves the original chapter structure for maps, histograms, Wasserstein+KDE, energy spectra, vertical profiles, and verification metrics.

This notebook required png figures, set `cfg["plotting"]["output_mode"]` to `"png"` or `"both"`.

In [ ]:
# Locate configuration relative to the notebook location (search upwards for config/example_config.yaml)
cfg_path = None

if config_path_str:
    candidate = Path(config_path_str)
    if not candidate.is_absolute():
        candidate = (Path.cwd().parent / candidate).resolve()
    if candidate.is_file():
        cfg_path = candidate
    else:
        print(f"Warning: Config file not found at {candidate}. Falling back to auto-discovery.")

if cfg_path is None:
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        candidate = base / "config" / "example_config.yaml"
        if candidate.is_file():
            cfg_path = candidate
            break

if cfg_path is None:
    raise FileNotFoundError(
        "Could not find config/example_config.yaml in cwd, parent, or grandparent directories."
    )

print(f"Using configuration: {cfg_path}")

# Load configuration via project helper to keep parity with CLI
cfg = _load_yaml(cfg_path)

# Project root is the parent of the config directory
try:
    # Find the index of the 'config' directory in the path
    config_idx = cfg_path.parts.index('config')
    # The project root is the path up to the 'config' directory
    project_root = Path(*cfg_path.parts[:config_idx]).resolve()
except ValueError:
    # Fallback: assume standard structure project/config/file.yaml if 'config' not found in path
    project_root = cfg_path.parent.parent.resolve()

# Resolve output_root from config (relative paths interpreted relative to the project root)
configured_out = cfg.get("paths", {}).get("output_root", "output/verification")
out_root = (project_root / configured_out).resolve()
out_root.mkdir(parents=True, exist_ok=True)
print(f"Project root: {project_root}")
print(f"Output root:  {out_root}")

In [ ]:
# Optional: Execute the full pipeline (data prep, metrics, plots) using CLI logic.
# This will generate all configured artifacts to out_root respecting cfg['modules'] flags
# and cfg['plotting']['output_mode'] (e.g., 'npz' to export numeric data only).
RUN_ALL = False  # Set to True to run everything end-to-end automatically.

if RUN_ALL:
    run_selected(cfg)
    print(f"All selected modules executed. Outputs under: {out_root}")

In [ ]:
# Prepare datasets once using the CLI API
# Returns: ds_target (Target), ds_prediction (Prediction), and standardized counterparts
(
    ds_target,
    ds_prediction,
    ds_target_std,
    ds_prediction_std,
) = prepare_datasets(cfg)

# Optionally choose a specific datetime and/or ensemble subset for map-like plots
(ds_target_plot, ds_prediction_plot) = _select_plot_datetime(
    ds_target, ds_prediction, cfg
)
(ds_prediction_plot, _ds_prediction_std_plot) = _select_plot_ensemble(
    ds_prediction_plot, ds_prediction_std, cfg
)

Loading target weatherbench data, originally stored in a Google Cloud Bucket, it
is already available on Alps. The content must match the config defined in the
first code cell.

Uncomment the localCluster to eagerly calculate and load the entire dataset into memory. Takes a long time but consequent cells will be faster thanks to the caching mechanism.

In [ ]:
# Target ground-truth loaded above in ds_target via prepare_datasets(cfg)
ds_target

Loading the prediction model output, expecting to have the same dimensions as the
target data. The content must match the config defined in the first code cell.

In [ ]:
# Prediction model outputs loaded above in ds_prediction via prepare_datasets(cfg)
ds_prediction

In [ ]:
# Optional missing-value check
CHECK_MISSING_VALUES = cfg["selection"].get("check_missing", False)
if CHECK_MISSING_VALUES:
    for var in ds_target.data_vars:
        missing = int(ds_target[var].isnull().sum())
        print(f"Ground Truth {var}: {missing} missing values")
    for var in ds_prediction.data_vars:
        missing = int(ds_prediction[var].isnull().sum())
        print(f"Prediction {var}: {missing} missing values")

In [ ]:
# Helper constants used by multiple plots
lat_bins = np.arange(-90, 91, 10)
n_bands = len(lat_bins) - 1
n_rows = n_bands // 2

## Maps
Generates spatial maps comparing the target (ERA5) and the model prediction.
Displays the fields side-by-side and the difference (bias).
If an ensemble is present, it can show individual members or the ensemble mean.

**Random Time Selection:** A random time step is selected to avoid bias in the comparison, ensuring that the assessment is representative of typical model performance.

**Consistent Color Scales:** By setting the same minimum and maximum values across both datasets for each variable, we ensure that color differences in the plots reflect true discrepancies, not artifacts of scaling.

**Spatial Patterns:** The plots reveal how the prediction model represents geographical features like weather fronts, high and low-pressure systems, and temperature gradients. Visual comparisons can immediately highlight areas where the prediction model performs well or poorly, guiding further investigation.

**Edge Effects:** Near the poles or data boundaries, artifacts may occur due to the coordinate system or data interpolation, potentially misleading the interpretation.

**Relevant Code:**
*   [`swissclim_evaluations.plots.maps.run`](../src/swissclim_evaluations/plots/maps.py#L27): Main entry point for map generation.

In [ ]:
# Display Maps
print("Displaying Maps...")
display_outputs(out_root / "maps", exclude_pattern="grid")

In [ ]:
# Multi-Lead Maps (Grids)
display_outputs(out_root / "maps", pattern_img="map_grid_*.png")

## Histograms
Compares the distribution of values for each variable between the target and prediction.
Useful for checking if the model captures the correct range and frequency of values.

**Relevant Code:**
*   [`swissclim_evaluations.plots.histograms.run`](../src/swissclim_evaluations/plots/histograms.py#L34): Computes and plots histograms.
*   [`_choose_edges`](../src/swissclim_evaluations/plots/histograms.py#L82): Determines common bin edges for fair comparison.

In [ ]:
print("Running Histograms module...")
display_outputs(out_root / "histograms", exclude_pattern="_grid_")

In [ ]:
# Multi-Lead Histograms (Grids)
display_outputs(out_root / "histograms", pattern_img="histogram_by_lead_*.png")

## Wasserstein Distance & KDE
Computes the Wasserstein distance between the target and prediction distributions.
Also plots Kernel Density Estimates (KDE) for a visual comparison of the distributions.
This is done on standardized data to allow comparison across variables.

**Relevant Code:**
*   [`swissclim_evaluations.plots.wd_kde.run`](../src/swissclim_evaluations/plots/wd_kde.py#L28): Computes Wasserstein distance and plots KDEs.

In [ ]:
print("Running Wasserstein Distance & KDE module...")
display_outputs(out_root / "wd_kde", exclude_pattern="ridgeline")

In [ ]:
# Temporal Evolution (Ridgeline)
display_outputs(out_root / "wd_kde", pattern_img="wd_kde_evolve_*.png", exclude_pattern=".csv")


## Energy Spectra
This chapter examines how energy is distributed across different spatial scales
in the atmosphere by computing and comparing the energy spectra of both datasets.
This analysis is critical in understanding the models' capabilities to simulate
atmospheric processes ranging from large-scale weather systems to small-scale
turbulence.

**FFT Computation:** The Fast Fourier Transform (FFT) is used to transform spatial data into the frequency domain, revealing how different scales contribute to the overall energy.

**Consideration of Earth's Curvature:** Adjustments are made for latitude to accurately calculate physical distances, which is essential for a meaningful spectrum analysis.

**Scale Representation:** The energy spectra show whether the Prediction captures the correct amount of energy at various spatial scales.

**Effective Resolution:** Identifying the effective resolution helps understand the smallest scales that the model can reliably simulate.

**Relevant Code:**
*   [`swissclim_evaluations.plots.energy_spectra.run`](../src/swissclim_evaluations/plots/energy_spectra.py#L533): Computes and plots kinetic energy spectra.
*   [`calculate_energy_spectra`](../src/swissclim_evaluations/plots/energy_spectra.py#L40): Performs the FFT and spectral analysis.

In [ ]:
print("Running Energy Spectra module...")
display_outputs(out_root / "energy_spectra", exclude_pattern="tripanel")

In [ ]:
# Spectrogram for Multi-Lead
display_outputs(out_root / "energy_spectra", pattern_img="energy_spectrogram_*tripanel.png", exclude_pattern=".csv")

## Vertical Profiles

In this chapter, the focus is on assessing how the relative error between the ML
model and ERA5 data varies with altitude across different latitude bands.
Vertical profiles are essential for understanding the atmospheric structure and
processes at different pressure levels. Obviously these plots only work for 3D
variables.

**Relative Error Calculation:** Using percentage differences provides a
normalized measure of error that is comparable across variables and pressure
levels.

**Latitude Band Analysis:** As with previous chapters, examining different
latitude bands acknowledges the variability in atmospheric conditions around the
globe.

**Altitude-Specific Insights:** The plots reveal whether the Prediction performs
consistently across different altitudes or if certain layers pose challenges.

**Atmospheric Dynamics:** Accurate representation of vertical profiles is
crucial for modeling phenomena like convection or the jet stream.

**Pressure Level Interpretation:** Lower pressure levels correspond to higher
altitudes. Inverted axes help represent this correctly but can be
counterintuitive.

**Relevant Code:**
*   [`swissclim_evaluations.metrics.vertical_profiles.run`](../src/swissclim_evaluations/metrics/vertical_profiles.py#L193): Computes vertical profile metrics.

In [ ]:
print("Running Vertical Profiles module...")
display_outputs(out_root / "vertical_profiles", exclude_pattern="evolution")

In [ ]:
# Temporal Evolution
display_outputs(out_root / "vertical_profiles", pattern_img="vertical_profile_evolution_*.png", exclude_pattern=".csv")


## Deterministic Metrics

The chapter consolidates various statistical metrics to provide a broad
evaluation of the prediction model's performance. By considering multiple metrics, we
gain a nuanced understanding of both the strengths and weaknesses of the model.

**Metric Diversity:** Including MAE, RMSE, MSE, Pearson correlation, and the
Fractions Skill Score (FSS) covers different aspects of model performance, from
average errors to spatial pattern accuracy.

**MAE, MSE and RMSE:** Offer insights into the average magnitude of errors, with
RMSE emphasizing larger discrepancies. The colors indicating high errors are
only implemented for these three metrics with standardization.

**Pearson Correlation:** Assesses the linear relationship, indicating whether
the model captures variability even if biases exist.

**FSS:** Evaluates spatial accuracy, which is particularly important for
predicting localized weather events.

**Standardization of Data:** Standardizing variables ensures that metrics are
not dominated by variables with larger magnitudes, allowing for fair
comparisons.

**Holistic Assessment:** The combination of metrics provides a comprehensive
performance profile, essential for model validation and comparison.

**Relevant Code:**
*   [`swissclim_evaluations.metrics.deterministic.run`](../src/swissclim_evaluations/metrics/deterministic.py#L483): Orchestrates metric calculation.
*   [`_calculate_all_metrics`](../src/swissclim_evaluations/metrics/deterministic.py#L136): Computes the scalar metrics.

In [ ]:
print("Running Deterministic Metrics module...")
display_outputs(out_root / "deterministic", exclude_pattern="line")

In [ ]:
# Temporal Evolution
display_outputs(out_root / "deterministic", pattern_img="det_line_*.png", exclude_pattern=".csv")


## Equitable Threat Score (ETS)
Calculates the Equitable Threat Score for precipitation or other threshold-based variables.
Measures the skill of the forecast relative to random chance.

Range: -1/3 to 1, where:
- 1 = perfect score
- 0 = no skill compared to random chance
- Negative values = worse than random chance

**Key Properties:**
- ETS measures how well predicted events correspond to observed events, accounting for hits due to random chance
- It's particularly useful for rare events (like precipitation above a high threshold)
- It's more equitable than the simple Threat Score because it accounts for hits that would occur by chance

**Relevant Code:**
*   [`swissclim_evaluations.metrics.ets.run`](../src/swissclim_evaluations/metrics/ets.py#L114): Computes ETS.

In [ ]:
print("Running ETS module...")
display_outputs(out_root / "ets", exclude_pattern="line")

In [ ]:
# Temporal Evolution
display_outputs(out_root / "ets", pattern_img="ets_line_*.png", exclude_pattern=".csv")

## SSIM

Structural Similarity Index Measure (SSIM) is a perceptual metric that quantifies image quality degradation. In the context of weather forecasting, it measures the structural similarity between the predicted and observed fields.

**Key Properties:**
- **Range:** -1 to 1, where 1 indicates perfect similarity.
- **Perceptual Quality:** Unlike MSE or MAE, SSIM considers changes in structural information, luminance, and contrast, aligning better with human visual perception of spatial patterns.
- **Spatial Context:** It evaluates the image quality locally using a sliding window, capturing local spatial dependencies.
- **Average SSIM:** An average SSIM score across all variables is provided to give a single summary statistic for the model's overall structural performance.

**Interpretation:**
- High SSIM values (close to 1) suggest that the model accurately captures the spatial structure and texture of the atmospheric fields.
- Low SSIM values indicate distortions in the spatial patterns, even if the pixel-wise errors (MAE/RMSE) are low.

**Relevant Code:**
*   [`swissclim_evaluations.metrics.ssim.run`](../src/swissclim_evaluations/metrics/ssim.py#L194): Computes SSIM for the datasets.
*   [`calculate_ssim`](../src/swissclim_evaluations/metrics/ssim.py#L23): Calculates SSIM for each variable individually.

In [ ]:
print("Running SSIM module...")
display_outputs(out_root / "ssim", exclude_pattern="lead_time")

In [ ]:
# SSIM — per-variable lines vs lead_time
display_outputs(out_root / "ssim", pattern_img="ssim_evolution_*.png", exclude_pattern=".csv")

## Bivariate Histograms

Comparison of bivariate densities (e.g. u10m vs v10m) using logarithmic contours.
- **Filled contours**: Reference (Ground Truth)
- **Line contours**: Prediction (Model)

**Key Properties:**
- **Logarithmic Scaling**: The use of logarithmic scaling for contour levels enhances the visibility of both high-density and low-density regions in the bivariate distribution.
- **Contour Representation**: Filled contours represent the ground truth data, while line contours represent the model predictions, allowing for a clear visual comparison between the two distributions.
- **Density Estimation**: The bivariate histograms provide insights into how well the model captures the joint distribution of two related variables, which is crucial for understanding inter-variable relationships.  

**Relevant Code:**
*   [`swissclim_evaluations.plots.bivariate_histograms.calculate_and_plot_bivariate_histograms`](../src/swissclim_evaluations/plots/bivariate_histograms.py#L38): Generates bivariate histograms.

In [ ]:
print("Displaying Bivariate plots...")
display_outputs(out_root / "multivariate")